# Tool Nodes and Tool Execution

Notebook **06** built a hand-written **`run_tools`** node that maps **`tool_calls`** to **`ToolMessage`** results. LangGraph ships **`ToolNode`** from **`langgraph.prebuilt`** — a production-ready executor with parallel-call support, error handling, and async tools. Pair it with **`tools_condition`** for the standard ReAct router.

This notebook covers **`ToolNode`**, **`tools_condition`**, parallel **`tool_calls`**, **`execute_tool_calls`** patterns, **`TOOLS_BY_NAME`** registries, tool error handling, **`invalid_tool_calls`**, custom vs prebuilt tool nodes, async tools, **`handle_tool_errors`**, and integration with the ReAct graph from **06**.

**Prerequisites:** **06. Agent Loops and the ReAct Pattern**, **01. LangChain/12** (tools), **04. Conditional Routing and Branching**.

**What you'll learn:**

- How **`ToolNode`** executes one or many **`tool_calls`** from the last **`AIMessage`**
- When to use **`tools_condition`** vs a custom router with step limits
- How tool errors become recoverable **`ToolMessage`** text instead of graph crashes
- How to extend the Notes API toolset and wire it into a compiled ReAct graph

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import langchain_core

try:
    import langgraph
    print("langgraph:", langgraph.__version__)
except ImportError:
    print("langgraph: pip install langgraph")

print("langchain-core:", langchain_core.__version__)

---

## 1. Why a Dedicated Tool Node?

The **tool node** is the **act** step in ReAct. It must:

1. Read **`tool_calls`** from the latest **`AIMessage`**
2. Invoke matching tools (possibly **in parallel**)
3. Return one **`ToolMessage`** per call with the correct **`tool_call_id`**
4. Surface errors as message text so the model can retry

```
AIMessage(tool_calls=[tc1, tc2, tc3])
         │
         ▼
    ┌──────────┐
    │ ToolNode │──► [ToolMessage(tc1), ToolMessage(tc2), ToolMessage(tc3)]
    └──────────┘
         │
         └── back to model node
```

| Approach | When to use |
|----------|-------------|
| **Hand-written `run_tools`** | Custom logging, auth gates, rate limits per tool |
| **`ToolNode(tools)`** | Standard execution — start here |
| **Wrapped `ToolNode`** | Add telemetry around prebuilt behavior |

---

## 2. Notes API Tool Corpus

Reuse the teaching corpus from **06** — FastAPI, Alembic, JWT, pytest:

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
    {"id": "c5", "text": "Use httpx.AsyncClient in FastAPI tests for async endpoints."},
]


@tool
def get_note(note_id: str) -> str:
    """Fetch a note by id (n1, n2, n3, n4)."""
    return NOTES.get(note_id, f"Unknown note id: {note_id}")


@tool
def search_docs(query: str) -> str:
    """Keyword search over documentation chunks c1-c5."""
    tokens = set(query.lower().split())
    hits = []
    for chunk in DOC_CHUNKS:
        if tokens & set(chunk["text"].lower().split()):
            hits.append(f"[{chunk['id']}] {chunk['text']}")
    return "\n".join(hits) if hits else "No matching chunks."


@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


tools = [get_note, search_docs, add]
TOOLS_BY_NAME = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

print("registered tools:", list(TOOLS_BY_NAME))

---

## 3. `ToolNode` from `langgraph.prebuilt`

**`ToolNode`** accepts a list of **`@tool`** callables or **`StructuredTool`** instances. It expects state with a **`messages`** key (e.g. **`MessagesState`**):

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

# Simulate what the model node would produce
fake_ai = AIMessage(
    content="",
    tool_calls=[{
        "name": "search_docs",
        "args": {"query": "alembic"},
        "id": "call_demo_1",
        "type": "tool_call",
    }],
)

result = tool_node.invoke({"messages": [fake_ai]})
tm = result["messages"][0]
print(type(tm).__name__, "id=", tm.tool_call_id)
print(tm.content)

**`ToolNode`** returns **`{"messages": [ToolMessage, ...]}`** — compatible with **`add_messages`** reducer (**02**).

---

## 4. `tools_condition` Router

**`tools_condition`** is the standard conditional edge after the model node:

| Last message | Route |
|--------------|-------|
| **`AIMessage`** with **`tool_calls`** | `"tools"` |
| Otherwise | **`END`** |

Equivalent to **`should_continue`** from **06**, without step-limit logic — wrap or replace for production.

In [ ]:
from langgraph.prebuilt import tools_condition

state_with_tools = {"messages": [fake_ai]}
state_done = {"messages": [AIMessage(content="JWT uses Bearer header [c3].")]}

print("with tool_calls:", tools_condition(state_with_tools))
print("final answer:   ", tools_condition(state_done))

---

## 5. Tool Message Contract

Every **`tool_call`** in an **`AIMessage`** must produce exactly one **`ToolMessage`**:

| Field | Rule |
|-------|------|
| **`tool_call_id`** | Must match **`tool_calls[i]["id"]`** |
| **`content`** | String (or serializable) observation for the model |
| **`name`** | Optional; helpful for debugging |

Mismatching ids causes the model to **ignore** tool output — a common silent failure.

---

## 6. Parallel `tool_calls` in One `AIMessage`

Modern models often emit **multiple** tool calls in a single turn. **`ToolNode`** executes all of them and returns multiple **`ToolMessage`** objects:

In [ ]:
parallel_ai = AIMessage(
    content="",
    tool_calls=[
        {"name": "search_docs", "args": {"query": "jwt"}, "id": "call_a", "type": "tool_call"},
        {"name": "get_note", "args": {"note_id": "n3"}, "id": "call_b", "type": "tool_call"},
        {"name": "add", "args": {"a": 7, "b": 5}, "id": "call_c", "type": "tool_call"},
    ],
)

parallel_result = tool_node.invoke({"messages": [parallel_ai]})
for msg in parallel_result["messages"]:
    print(f"{msg.tool_call_id}: {str(msg.content)[:60]}...")

Hand-written nodes should loop **`ai_message.tool_calls`** the same way — never assume a single call.

---

## 7. `execute_tool_calls` and `TOOLS_BY_NAME`

The manual pattern from **06** and **01. LangChain/12** — useful when you need custom behavior:

In [ ]:
def execute_tool_calls(ai_message: AIMessage) -> list[ToolMessage]:
    results = []
    for tc in ai_message.tool_calls:
        tool = TOOLS_BY_NAME.get(tc["name"])
        try:
            if tool is None:
                content = f"Unknown tool: {tc['name']}"
            else:
                content = str(tool.invoke(tc["args"]))
        except Exception as exc:
            content = f"Tool error: {exc}"
        results.append(ToolMessage(content=content, tool_call_id=tc["id"]))
    return results


manual_messages = execute_tool_calls(parallel_ai)
print(f"manual produced {len(manual_messages)} ToolMessages")
print("ids match:", [m.tool_call_id for m in manual_messages])

**`TOOLS_BY_NAME`** is the registry pattern — O(1) lookup by tool name from model output.

---

## 8. Tool Error Handling → `ToolMessage`

Tools should **not** crash the graph. Exceptions become **`ToolMessage`** content so the model can adjust:

In [ ]:
@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


error_tools = tools + [divide]
error_node = ToolNode(error_tools)

bad_call = AIMessage(
    content="",
    tool_calls=[{
        "name": "divide",
        "args": {"a": 10, "b": 0},
        "id": "call_err",
        "type": "tool_call",
    }],
)

err_result = error_node.invoke({"messages": [bad_call]})
print(err_result["messages"][0].content[:120])

The model sees the error string on the next turn and can explain or retry with different args.

---

## 9. `invalid_tool_calls`

When the model emits **malformed** tool JSON, LangChain stores them on **`AIMessage.invalid_tool_calls`** instead of **`tool_calls`**. A robust tool node handles both:

In [ ]:
# Constructed example — mirrors what the parser produces on bad JSON
ai_with_invalid = AIMessage(
    content="",
    tool_calls=[],
    invalid_tool_calls=[{
        "name": "get_note",
        "args": "not-valid-json",
        "id": "call_bad",
        "type": "invalid_tool_call",
        "error": "JSON decode error",
    }],
)

print("valid calls:", ai_with_invalid.tool_calls)
print("invalid calls:", ai_with_invalid.invalid_tool_calls)


def feedback_invalid_calls(ai_message: AIMessage) -> list[ToolMessage]:
    return [
        ToolMessage(
            content=f"Invalid tool call for {itc.get('name')}: {itc.get('error')}",
            tool_call_id=itc["id"],
        )
        for itc in ai_message.invalid_tool_calls or []
    ]


invalid_feedback = feedback_invalid_calls(ai_with_invalid)
print(invalid_feedback[0].content)

**`ToolNode`** handles valid calls; for **`invalid_tool_calls`**, add a thin wrapper or custom node that feeds back parse errors before returning to the model.

---

## 10. Custom Tool Node vs `ToolNode`

Wrap **`ToolNode`** when you need auth, audit logs, or metrics — keep execution logic in the prebuilt node:

In [ ]:
AUDIT_LOG: list[str] = []


def audited_tool_node(state: dict) -> dict:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        for tc in last.tool_calls:
            AUDIT_LOG.append(f"invoke {tc['name']} args={tc['args']}")
    return ToolNode(tools).invoke(state)


AUDIT_LOG.clear()
audited_result = audited_tool_node({"messages": [parallel_ai]})
print("audit:", AUDIT_LOG)
print("tool messages:", len(audited_result["messages"]))

Prefer **composition** over reimplementing invoke logic.

---

## 11. Async Tools Preview

**`@tool`** supports **`async def`**. **`ToolNode`** dispatches async tools when the graph runs with **`ainvoke`** / **`astream`** (**13**):

In [ ]:
import asyncio


@tool
async def async_search_docs(query: str) -> str:
    """Async keyword search over documentation chunks."""
    await asyncio.sleep(0.01)  # simulate I/O
    return search_docs.invoke({"query": query})


async_tools = [get_note, async_search_docs, add]
async_tool_node = ToolNode(async_tools)

async_ai = AIMessage(
    content="",
    tool_calls=[{
        "name": "async_search_docs",
        "args": {"query": "fastapi httpx"},
        "id": "call_async",
        "type": "tool_call",
    }],
)

# Sync invoke still works for quick demos
async_result = async_tool_node.invoke({"messages": [async_ai]})
print(async_result["messages"][0].content)

In FastAPI services (**16**), compile once and use **`await graph.ainvoke(...)`** for async tool I/O.

---

## 12. `ToolNode` with `handle_tool_errors`

Pass **`handle_tool_errors=True`** (default) or a custom template string to control error message formatting:

In [ ]:
safe_node = ToolNode(
    error_tools,
    handle_tool_errors="Tool {tool_name} failed: {error}",
)

safe_result = safe_node.invoke({"messages": [bad_call]})
print(safe_result["messages"][0].content)

Set **`handle_tool_errors=False`** only if you want exceptions to propagate (rare — prefer guarded nodes in **14**).

---

## 13. Integration with ReAct Graph from 06

Replace hand-written **`run_tools`** with **`ToolNode`** + **`tools_condition`**. Keep **`step_count`** in a custom router:

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    step_count: int


SYSTEM = """You are a Notes API assistant.
Use search_docs and get_note before answering factual questions.
Use add for arithmetic. Cite chunk ids like [c2] when possible."""

MAX_STEPS = 8


def call_model(state: AgentState) -> dict:
    messages = [{"role": "system", "content": SYSTEM}] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response], "step_count": state.get("step_count", 0) + 1}


def route_with_limit(state: AgentState):
    if state.get("step_count", 0) >= MAX_STEPS:
        return END
    return tools_condition(state)


def build_toolnode_agent():
    builder = StateGraph(AgentState)
    builder.add_node("model", call_model)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "model")
    builder.add_conditional_edges("model", route_with_limit)
    builder.add_edge("tools", "model")
    return builder.compile()


agent = build_toolnode_agent()

out = agent.invoke({
    "messages": [HumanMessage(content="What pytest file holds DB fixtures? Use tools.")],
    "step_count": 0,
})

for msg in out["messages"]:
    if isinstance(msg, AIMessage) and msg.tool_calls:
        print("ai calls:", [tc["name"] for tc in msg.tool_calls])
    elif isinstance(msg, ToolMessage):
        print("tool:", str(msg.content)[:70])
print("final:", out["messages"][-1].content[:120] if out["messages"][-1].content else "(tool round)")

Same topology as **06** — only the tool node and router changed.

---

## 14. Extending Tools — `bind_tools` Refresh

When you add tools, update **three** places: the list, **`TOOLS_BY_NAME`**, and **`bind_tools`**:

In [ ]:
@tool
def list_note_ids() -> str:
    """Return all available note ids."""
    return ", ".join(sorted(NOTES))


extended_tools = tools + [list_note_ids]
extended_by_name = {t.name: t for t in extended_tools}
llm_extended = llm.bind_tools(extended_tools)

print("extended:", [t.name for t in extended_tools])
print("bind_tools schema count:", len(llm_extended.kwargs.get("tools", [])))

Rebuild **`ToolNode(extended_tools)`** and recompile the graph — compile once at startup (**16**).

---

## 15. Design Guidelines

| Guideline | Rationale |
|-----------|----------|
| **Start with `ToolNode`** | Battle-tested parallel execution and error wrapping |
| **Keep `TOOLS_BY_NAME` in sync** | Manual nodes and tests need the same registry |
| **One `ToolMessage` per `tool_call`** | Required for model to correlate observations |
| **Errors → message text** | Let the model recover; don't kill the thread |
| **Handle `invalid_tool_calls`** | Malformed JSON happens in production |
| **Cap steps in router** | `tools_condition` alone has no limit |
| **Async tools + `ainvoke`** | Don't block the event loop in FastAPI (**16**) |
| **Audit at the wrapper** | Log invocations without forking `ToolNode` internals |

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Tool not in `ToolNode(tools)` list | "Unknown tool" message | Register all `bind_tools` tools |
| Mismatched `tool_call_id` | Model ignores result | Copy `id` from `tool_calls` |
| Only handling first `tool_call` | Silent data loss | Loop all calls |
| Raising in tool node | Graph crash | `handle_tool_errors=True` |
| Forgetting `tools → model` edge | One-shot tools | Add cycle edge |
| `bind_tools` out of sync with `ToolNode` | Model calls missing tools | Update both together |
| Ignoring `invalid_tool_calls` | Stuck loop with no observations | Feed back parse errors |
| Recompile per request | Slow cold starts | Singleton graph (**16**) |

---

## 17. Summary

**`ToolNode`** is the standard executor for ReAct graphs: it reads **`tool_calls`** from the last **`AIMessage`**, invokes registered tools (including **parallel** calls), and returns **`ToolMessage`** observations. Pair it with **`tools_condition`** for routing and wrap with **step limits** for production safety.

Key takeaways:

- **`execute_tool_calls`** + **`TOOLS_BY_NAME`** explain what **`ToolNode`** does internally — customize via wrappers, not duplication.
- Tool **errors** and **`invalid_tool_calls`** should become **`ToolMessage`** text so the model can recover.
- **`handle_tool_errors`** customizes error strings; **async tools** need **`ainvoke`** in async servers.
- The **06** ReAct graph drops in **`ToolNode(tools)`** with minimal changes.

Demonstrations registered Notes API tools, ran **`ToolNode`** standalone, compared manual execution, handled errors and invalid calls, previewed async tools, and compiled a full ToolNode-based agent.

Next: **08. Checkpointing and Thread Persistence** — persist multi-turn state with **`MemorySaver`**, **`thread_id`**, and durable backends.